# Session1_Task3 — Sales Trend Analysis

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker              # กำหนดรูปแบบตัวเลขบนแกน เช่น เพิ่ม $
from matplotlib.backends.backend_pdf import PdfPages  # บันทึกหลาย figure เป็น PDF เดียว

s = pd.read_csv('sales_transactions_cleaned.csv')

# pd.to_datetime() → แปลง string เป็น datetime
s['date']    = pd.to_datetime(s['date'], errors='coerce')
# .dt.to_period('M') → ตัดเวลาทิ้ง เหลือแค่ปี-เดือน เช่น 2024-03
s['month']   = s['date'].dt.to_period('M').astype(str)
# revenue = (quantity × price) - discount
s['revenue'] = (s['quantity'] * s['price']) - pd.to_numeric(s['discount_amount'], errors='coerce').fillna(0)

In [2]:
# --- คำนวณ Monthly Stats ---

# .groupby() → จัดกลุ่มตามเดือน
# .agg()     → ทำหลายการคำนวณพร้อมกัน
#   rev = รวม revenue ทั้งเดือน
#   tx  = นับ transaction ที่ไม่ซ้ำ
# .assign()  → เพิ่มคอลัมน์ aov = average order value
m = (s.groupby('month')
      .agg(rev=('revenue',        'sum'),
           tx= ('transaction_id', 'nunique'))
      .assign(aov=lambda x: x['rev'] / x['tx'])
      .sort_index().reset_index())

display(m)

,month,rev,tx,aov
0,2023-11,62.40,10,6.240000
1,2023-12,10549.62,1350,7.814533
2,2024-01,10264.12,1364,7.525015
3,2024-02,9281.58,1175,7.899217
4,2024-03,12047.21,1244,9.684252
5,2024-04,11623.09,1199,9.693987
6,2024-05,12012.67,1177,10.206177
7,2024-06,11173.00,1103,10.129646
8,2024-07,69380.44,1092,63.535201
9,2024-08,3073.98,26,118.230000


In [3]:
# --- Top 3 months ---

# .nlargest(3,'rev') → เลือก 3 แถวที่มีค่า rev มากที่สุด
# f'${v:,.2f}'       → จัดรูปแบบ: , = จุลภาคทุก 3 หลัก, .2f = ทศนิยม 2 ตำแหน่ง
t3 = (m.nlargest(3, 'rev')[['month', 'rev']]
       .reset_index(drop=True)
       .assign(rev=lambda x: x['rev'].apply(lambda v: f'${v:,.2f}')))

display(t3)

,month,rev
0,2024-07,"$69,380.44"
1,2024-03,"$12,047.21"
2,2024-05,"$12,012.67"


In [4]:
# --- สร้าง PDF ---

metrics = [
    ('rev', 'tomato',    'Total Sales Revenue ($)'),
    ('tx',  'steelblue', 'Number of Transactions'),
    ('aov', 'seagreen',  'Average Order Value ($)'),
]

with PdfPages('Session1_SalesTrends.pdf') as pdf:

    for col, color, title in metrics:
        fig, ax = plt.subplots(figsize=(10, 5))
        # ax.plot() → วาด line chart
        #   marker='o' → ใส่จุดกลมที่ทุก data point
        ax.plot(m['month'], m[col], marker='o', color=color, linewidth=2, markersize=5)
        ax.set_title(title, fontsize=14, fontweight='bold', pad=12)
        ax.set_xlabel('Month')
        # rotation=45 → หมุน label แกน x 45 องศา ไม่ให้ทับกัน
        ax.tick_params(axis='x', rotation=45)
        ax.grid(axis='y', linestyle='--', alpha=0.5)
        # FuncFormatter → กำหนดรูปแบบตัวเลขบนแกน y
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(
            lambda v, _: f'${v:,.0f}' if col != 'tx' else f'{int(v):,}'
        ))
        fig.tight_layout()
        # pdf.savefig() → บันทึก figure นี้เป็นหน้าใหม่ใน PDF
        pdf.savefig(fig, bbox_inches='tight'); plt.close()

    # Top 3 Table
    fig, ax = plt.subplots(figsize=(6, 2.5))
    ax.axis('off')  # ซ่อน axes เพื่อให้เห็นแค่ตาราง
    ax.set_title('Top 3 Months by Sales Revenue', fontsize=13, fontweight='bold', pad=16)
    tbl = ax.table(cellText=t3.values, colLabels=['Month', 'Total Revenue'],
                   loc='center', cellLoc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.5, 2.2)
    # วนลูปจัด style แต่ละ cell: r==0 คือแถว header
    for (r, _), cell in tbl.get_celld().items():
        if r == 0: cell.set_facecolor('steelblue'); cell.set_text_props(color='white', fontweight='bold')
        cell.set_edgecolor('lightgray')
    fig.tight_layout()
    pdf.savefig(fig, bbox_inches='tight'); plt.close()

print('✅ Saved Session1_SalesTrends.pdf')

# === จุดสังเกต ===
# ✔ PDF มี 4 หน้า (3 line charts + 1 table)
# ✔ แกน y ของ revenue/aov มีเครื่องหมาย $
# ✔ top 3 table มี 3 แถว

✅ Saved Session1_SalesTrends.pdf
